**Expected Results of GVAE-CV on MNIST**


1.   Reconstruction(BCE)=69.94
2.   log p(x) = -106.41
3.   Err(%) = 17.25
4.   M-MSE=20.37

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import argparse
import sys, getopt as gopt, time
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import subprocess
import os
import pickle

from density.fit_gmm import fit_gmm
from density.eval_logpx import evaluate_logpx



In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=360, latent_dim=20):
        super(Encoder, self).__init__()


        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)

        self._init_weights()

    def _init_weights(self, sigma=0.1):
        for layer in [self.fc1, self.fc2, self.fc3, self.fc_mu]:
            nn.init.normal_(layer.weight, mean=0.0, std=sigma)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        mu = self.fc_mu(x)
        return mu


class Decoder(nn.Module):
    def __init__(self, latent_dim=20, hidden_dim=360, output_dim=784,l2_lambda=1e-3):
        super(Decoder, self).__init__()


        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        self._init_weights()

    def _init_weights(self, sigma=0.1):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.normal_(layer.weight, mean=0.0, std=sigma)
            nn.init.constant_(layer.bias, 0.0)

    def compute_l2_penalty(self):
        l2_penalty = 0
        for param in self.decoder.parameters():
            if param.requires_grad:
                l2_penalty += torch.sum(param**2)
        return self.l2_lambda * l2_penalty

    def forward(self, z):
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        z = torch.sigmoid(self.fc3(z))
        return z
    def compute_loss(self, x, recon_x):
        reconstruction_loss = self.reconstruction_loss(recon_x, x)
        l2_penalty = self.compute_l2_penalty()
        return -(reconstruction_loss + l2_penalty)


# GVAE-CV
class GVAE_CV(nn.Module):
    def __init__(
        self,
        input_dim=784,
        hidden_dim=360,
        latent_dim=20,
        fixed_variance=torch.tensor(0.0),
    ):
        super(GVAE_CV, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)
        self.fixed_variance = fixed_variance

    def reparameterize(self, mu):
        std = torch.exp(0.5 * self.fixed_variance)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        mu = self.encoder(x)
        z = self.reparameterize(mu)
        recon_x = self.decoder(z)
        return recon_x, mu

In [3]:

class NumpyDataset(Dataset):
    def __init__(self, dataX, dataY=None):
        self.dataX = np.load(dataX)
        self.dataY = np.load(dataY) if dataY is not None else None

    def __len__(self):
        return len(self.dataX)

    def __getitem__(self, idx):
        data = torch.tensor(self.dataX[idx], dtype=torch.float32)
        label = torch.tensor(self.dataY[idx], dtype=torch.long) if self.dataY is not None else None
        return data, label

# Set up argument parser
parser = argparse.ArgumentParser(description='Process some datasets.')
parser.add_argument('--dataX', type=str, default="/Users/bethtassew/Desktop/mnist/trainX.npy", help='Path to training data X')
parser.add_argument('--dataY', type=str, default="/Users/bethtassew/Desktop/mnist/trainY.npy", help='Path to training data Y')
parser.add_argument('--devX', type=str, default="/Users/bethtassew/Desktop/mnist/validX.npy", help='Path to development data X')
parser.add_argument('--devY', type=str, default="/Users/bethtassew/Desktop/mnist/validY.npy", help='Path to development data Y')
parser.add_argument('--testX', type=str, default="/Users/bethtassew/Desktop/mnist/testX.npy", help='Path to test data X')
parser.add_argument('--testY', type=str, default="/Users/bethtassew/Desktop/mnist/testY.npy", help='Path to test data Y')
parser.add_argument('--verbosity', type=int, default=0, help='Verbosity level')

# Parse the arguments, ignoring unrecognized ones
args, unknown = parser.parse_known_args()

# Access the arguments
dataX = args.dataX
dataY = args.dataY
devX = args.devX
devY = args.devY
testX = args.testX
testY = args.testY
verbosity = args.verbosity

# Create datasets and dataloaders
train_dataset = NumpyDataset(dataX, dataY)
train_loader = DataLoader(dataset=train_dataset, batch_size=200, shuffle=True)

dev_dataset = NumpyDataset(devX, devY)
dev_loader = DataLoader(dataset=dev_dataset, batch_size=200, shuffle=False)

test_dataset = NumpyDataset(testX, testY)
test_loader = DataLoader(dataset=test_dataset, batch_size=200, shuffle=False)

# Print the paths and datasets for verification
print("Train-set: X: {} | Y: {}".format(dataX, dataY))
print("  Dev-set: X: {} | Y: {}".format(devX, devY))
print("  Test-set: X: {} | Y: {}".format(testX, testY))


Train-set: X: /Users/bethtassew/Desktop/mnist/trainX.npy | Y: /Users/bethtassew/Desktop/mnist/trainY.npy
  Dev-set: X: /Users/bethtassew/Desktop/mnist/validX.npy | Y: /Users/bethtassew/Desktop/mnist/validY.npy
  Test-set: X: /Users/bethtassew/Desktop/mnist/testX.npy | Y: /Users/bethtassew/Desktop/mnist/testY.npy


In [4]:

def rescale_gradients(model, max_norm=5.0):
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)


def train_model(model, train_loader):

    optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

    model.train()
    for epoch in range(50):
        total_loss = 0
        total_samples = 0
        for batch_idx, (data, _) in enumerate(train_loader):

            data = (data > 0.5).float()
            data = data.view(data.size(0), -1)
            optimizer.zero_grad()

            recon_data, mu = model(data)
            recon_data = recon_data.view(recon_data.size(0), -1)
            loss = totalloss(recon_data, data, mu, fixed_variance)
            loss.backward()
            rescale_gradients(model)
            optimizer.step()
            total_loss += loss.item()
            total_samples += data.size(0)
        print(f"Epoch {epoch + 1}, Total_Loss: {total_loss / len(train_loader.dataset):.4f}")


# M-MSE Loss
def masked_mse_loss(model, loader):
    model.eval()
    total_mse = 0.0
    total_samples = 0
    total_masked_elements = 0
    with torch.no_grad():
        for data, _ in loader:

            data = data.view(data.size(0), -1)
            data = (data > 0.5).float()
            mask = torch.ones_like(data, dtype=torch.bool)
            mask[:, : data.size(1) // 2] = 0

            masked_data = data * mask.float()
            masked_data = (masked_data > 0.5).float()
            reconstructed, _ = model(masked_data)
            reconstructed = reconstructed.view(data.size(0), -1)

            mse = F.mse_loss(reconstructed[~mask], data[~mask], reduction="sum")
            total_mse += mse.item() * data.size(0)
            total_samples += data.size(0)
            total_masked_elements += (~mask).sum().item()

    avg_mse = total_mse / (total_samples * data.size(1) // 2)

    return avg_mse

# BCE Loss
def bce_loss(model, loader):
    model.eval()
    total_bce = 0.0
    total_samples = 0
    with torch.no_grad():
        for data, _ in loader:
            data = data.view(data.size(0), -1)
            data = (data > 0.5).float()
            recon_data, _ = model(data)
            recon_data = recon_data.view(data.size(0), -1)

            bce = F.binary_cross_entropy(recon_data, data, reduction="sum")
            total_bce += bce.item()

            total_samples += data.size(0)

    # Normalize by the total number of elements
    avg_bce = total_bce / total_samples

    return avg_bce

def totalloss(recon_x, x, mu, fixed_variance):
    recon_x = recon_x.view(recon_x.size(0), -1)
    x = x.view(x.size(0), -1)
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + fixed_variance - mu.pow(2) - torch.exp(fixed_variance))
    kl_loss = kl_loss / 20
    return recon_loss + kl_loss


# Classification error
def classification_error(model, data_loader, latent_dim, num_classes):
    model.eval()
    latent_representations = []
    labels = []

    with torch.no_grad():
        for batch in data_loader:
            data, target = batch

            data = data.view(data.size(0), -1)


            if target.ndim > 1:
                target = torch.argmax(target, dim=1)

            mu = model.encoder(data)
            latent_representations.append(mu.cpu().numpy())
            labels.append(target.cpu().numpy())

    X = np.vstack(latent_representations)
    y = np.hstack(labels)


    assert (
        X.shape[0] == y.shape[0]
    ), "Mismatch in the number of samples between X and y!"

    classifier = LogisticRegression(max_iter=1000, multi_class="multinomial")
    classifier.fit(X, y)

    y_pred = classifier.predict(X)
    accuracy = accuracy_score(y, y_pred)

    error_percentage = 100 * (1 - accuracy)
    return error_percentage


# GMM
# def fit_gmm(latent_vectors, n_components=75):
#     gmm = GaussianMixture(
#         n_components=n_components, covariance_type="full", random_state=42
#     )
#     gmm.fit(latent_vectors)
#     return gmm


# log likelihood
# def monte_carlo_log_likelihood(gmm, gvae, data_loader, n_samples=5000):

#     gmm_samples, _ = gmm.sample(n_samples)
#     z_samples = torch.tensor(gmm_samples, dtype=torch.float32)

#     log_p_z = gmm.score_samples(gmm_samples)
#     log_p_x_given_z = []


#     with torch.no_grad():
#         for i in range(0, n_samples, data_loader.batch_size):
#             batch_z = z_samples[i : i + data_loader.batch_size]
#             recon_x = gvae.decoder(batch_z)

#             log_p_x_given_z.extend(
#                 -torch.nn.functional.binary_cross_entropy(
#                     recon_x, recon_x, reduction="none"
#                 )
#                 .sum(dim=1)
#                 .cpu()
#                 .numpy()
#             )

#     log_p_x_given_z = np.array(log_p_x_given_z)
#     log_likelihood = np.mean(log_p_z + log_p_x_given_z)

#     return log_likelihood


# def final_modeling(model, loader):
#     latent_vectors = []
#     model.eval()

#     with torch.no_grad():
#         for batch_idx, (data, _) in enumerate(
#             tqdm(train_loader, desc="Processing Latent Vectors")
#         ):
#             data = data.view(data.size(0), -1)
#             mu = model.encoder(data)
#             latent_vectors.append(mu.cpu().numpy())
#     latent_vectors = np.vstack(latent_vectors)


#     # Fit the GMM
#     gmm = fit_gmm(latent_vectors, n_components=75)

#     # Calculate log likelihood
#     log_likelihood = monte_carlo_log_likelihood(gmm, model, train_loader)


#     return log_likelihood


input_dim = 784
latent_dim = 20
hidden_dim = 360
fixed_variance = torch.tensor(0.0)
gvae = GVAE_CV(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim,
    fixed_variance=fixed_variance
)

sim_start_time = time.time()
print("--------------- Training ---------------")
train_model(gvae, train_loader)

sim_time = time.time() - sim_start_time
print(f"Training Time = {sim_time:.4f} seconds")

print("--------------- Testing ---------------")
bce_loss = bce_loss(gvae, test_loader)
classification_error = classification_error(
    gvae, test_loader, latent_dim=latent_dim, num_classes=10
)
gmm = fit_gmm(train_loader, gvae, latent_dim=latent_dim, n_components=75)
log_likelihood = evaluate_logpx(test_loader, gvae, gmm, latent_dim=latent_dim, num_samples=5000)


masked_mse = masked_mse_loss(gvae, test_loader)
# log_likelihood = final_modeling(gvae, test_loader)

print(
    f"Test M-MSE: {masked_mse:.4f}, Test BCE: {bce_loss:.4f}, Error(%): {classification_error:.2f}%, log p(x): {log_likelihood}"
)


--------------- Training ---------------
Epoch 1, Total_Loss: 264.3794
Epoch 2, Total_Loss: 165.9831
Epoch 3, Total_Loss: 143.8005
Epoch 4, Total_Loss: 131.3907
Epoch 5, Total_Loss: 123.0194
Epoch 6, Total_Loss: 116.6279
Epoch 7, Total_Loss: 111.5762
Epoch 8, Total_Loss: 107.3880
Epoch 9, Total_Loss: 103.9687
Epoch 10, Total_Loss: 101.0639
Epoch 11, Total_Loss: 98.5747
Epoch 12, Total_Loss: 96.3913
Epoch 13, Total_Loss: 94.4690
Epoch 14, Total_Loss: 92.7359
Epoch 15, Total_Loss: 91.1490
Epoch 16, Total_Loss: 89.7465
Epoch 17, Total_Loss: 88.4706
Epoch 18, Total_Loss: 87.3184
Epoch 19, Total_Loss: 86.2193
Epoch 20, Total_Loss: 85.2326
Epoch 21, Total_Loss: 84.3142
Epoch 22, Total_Loss: 83.4302
Epoch 23, Total_Loss: 82.6280
Epoch 24, Total_Loss: 81.8679
Epoch 25, Total_Loss: 81.1501
Epoch 26, Total_Loss: 80.4470
Epoch 27, Total_Loss: 79.8052
Epoch 28, Total_Loss: 79.1587
Epoch 29, Total_Loss: 78.5856
Epoch 30, Total_Loss: 78.0368
Epoch 31, Total_Loss: 77.4979
Epoch 32, Total_Loss: 76.974

/Users/bethtassew/Desktop/GVAE-CV/gvae/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Extracting latent vectors from dataLoader using the provided model...
Collected latent data shape: torch.Size([50000, 20])
Saving GMM to file: gmm.pkl
Test M-MSE: 22.9143, Test BCE: 68.0074, Error(%): 10.01%, log p(x): -163.79117286834716
